In [ ]:
# Bootstrap defaults for running in VS Code
import os, glob, psutil

# Ensure SPARK_HOME is set (fallback to local build dist)
SPARK_HOME = os.environ.get("SPARK_HOME", "/users/chenqh23/spark/dist")
os.environ["SPARK_HOME"] = SPARK_HOME
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'

for p in psutil.process_iter(['pid','name','cmdline']):
    cl = ' '.join(p.info.get('cmdline') or [])
    if p.info.get('name') == 'java' and 'org.apache.spark.deploy.SparkSubmit' in cl:
        try: p.kill()
        except: pass
# ensure classpath
try:
    import findspark; findspark.init(os.environ['SPARK_HOME'])
except Exception:
    pass

# Use local mode unless a cluster URL is provided
os.environ.setdefault("SPARK_MASTER_URL", "local[*]")

# Data and event log defaults
os.environ.setdefault("DATA_ROOT", "/users/chenqh23/spark-rapids-examples/datasets")
os.environ.setdefault("EVENTLOG_DIR", "/tmp/spark-events")
try:
    os.makedirs(os.environ["EVENTLOG_DIR"], exist_ok=True)
except Exception:
    pass

# Auto-detect RAPIDS jar in SPARK_HOME/jars if not provided
if "RAPIDS_JAR" not in os.environ:
    jars_dir = os.path.join(SPARK_HOME, "jars")
    candidates = []
    for pattern in ("rapids-4-spark_*.jar", "rapids-4-spark*.jar"):
        candidates.extend(glob.glob(os.path.join(jars_dir, pattern)))
    rapids_jar = next((p for p in candidates if p.endswith(".jar")), None)
    if rapids_jar:
        os.environ["RAPIDS_JAR"] = rapids_jar

print("SPARK_HOME =", os.environ["SPARK_HOME"]) 
print("SPARK_MASTER_URL =", os.environ["SPARK_MASTER_URL"]) 
print("EVENTLOG_DIR =", os.environ["EVENTLOG_DIR"]) 
print("RAPIDS_JAR =", os.environ.get("RAPIDS_JAR", "<auto> or jars in SPARK_HOME"))


SPARK_HOME = /users/chenqh23/spark/dist
SPARK_MASTER_URL = local[*]
EVENTLOG_DIR = /tmp/spark-events
RAPIDS_JAR = /users/chenqh23/spark/dist/jars/rapids-4-spark_2.12-25.08.0.jar


# Microbenchmarks on GPU
This is a notebook for microbenchmarks running on GPU. 

In [ ]:
from pyspark.sql import SparkSession
from pyspark.conf import SparkConf
import time
import os
# Change to your cluster ip:port and directories
SPARK_MASTER_URL = os.getenv("SPARK_MASTER_URL", "spark:your-ip:port")
RAPIDS_JAR = os.getenv("RAPIDS_JAR", "/your-path/rapids-4-spark_2.12-25.08.0.jar")


Run the microbenchmark with retryTimes

In [3]:
# Robust Spark startup to fix Py4J "Answer from Java side is empty"
import os, psutil
from pyspark.sql import SparkSession
from pyspark.conf import SparkConf

# Environment
os.environ['JAVA_HOME'] = os.environ.get('JAVA_HOME', '/usr/lib/jvm/java-17-openjdk-amd64')
os.environ['SPARK_HOME'] = os.environ.get('SPARK_HOME', '/users/chenqh23/spark/dist')
os.environ.setdefault('SPARK_MASTER_URL', 'local[*]')

# Stop an existing Spark and kill orphan Spark JVMs
try:
    spark.stop()
except Exception:
    pass
try:
    for p in psutil.process_iter(['pid','name','cmdline','username']):
        cl = ' '.join(p.info.get('cmdline') or [])
        if p.info.get('name') == 'java' and 'org.apache.spark.deploy.SparkSubmit' in cl:
            try:
                p.kill()
            except Exception:
                pass
except Exception:
    pass

# Java 17 add-opens flags
_DEF_OPENS = (
    "--add-opens=java.base/java.lang=ALL-UNNAMED "
    "--add-opens=java.base/java.lang.invoke=ALL-UNNAMED "
    "--add-opens=java.base/java.lang.reflect=ALL-UNNAMED "
    "--add-opens=java.base/java.io=ALL-UNNAMED "
    "--add-opens=java.base/java.net=ALL-UNNAMED "
    "--add-opens=java.base/java.nio=ALL-UNNAMED "
    "--add-opens=java.base/java.util=ALL-UNNAMED "
    "--add-opens=java.base/java.util.concurrent=ALL-UNNAMED "
    "--add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED "
    "--add-opens=java.base/jdk.internal.ref=ALL-UNNAMED "
    "--add-opens=java.base/sun.nio.ch=ALL-UNNAMED "
    "--add-opens=java.base/sun.nio.cs=ALL-UNNAMED "
    "--add-opens=java.base/sun.security.action=ALL-UNNAMED "
    "--add-opens=java.base/sun.util.calendar=ALL-UNNAMED"
)

# Build base conf
base = (SparkConf()
    .setMaster("local[*]")
    .setAppName("Microbenchmark on GPU")
    .set("spark.driver.memory", "12g")
    .set("spark.sql.adaptive.enabled", "true")
    .set("spark.sql.files.maxPartitionBytes", "128m")
    .set("spark.sql.shuffle.partitions", "96")
    .set("spark.locality.wait", "0")
    .set("spark.scheduler.mode", "FAIR")
    .set("spark.eventLog.enabled", "false")
    .set("spark.driver.extraJavaOptions", _DEF_OPENS)
    .set("spark.executor.extraJavaOptions", _DEF_OPENS)
)

# GPU settings (avoid duplicate JARs; rely on SPARK_HOME/jars)
gpu = (base
    .set("spark.plugins", "com.nvidia.spark.SQLPlugin")
    .set("spark.rapids.sql.enabled", "true")
    .set("spark.rapids.sql.allowMultipleJars", "ALWAYS")
    .set("spark.rapids.sql.concurrentGpuTasks", "3")
    .set("spark.rapids.sql.batchSizeBytes", "256m")
    .set("spark.rapids.sql.multiThreadedRead.numThreads", "16")
    .set("spark.rapids.sql.exec.CollectLimitExec", "true")
    # Enable GPU-accelerated cached table scans
    .set("spark.sql.cache.serializer", "com.nvidia.spark.ParquetCachedBatchSerializer")
    .set("spark.rapids.sql.exec.InMemoryTableScanExec", "true")
    .set("spark.rapids.memory.gpu.maxAllocFraction", "0.5")
    .set("spark.rapids.memory.gpu.allocFraction", "0.25")
    .set("spark.rapids.memory.gpu.minAllocFraction", "0.0")
    .set("spark.rapids.memory.gpu.reserve", "2G")
    .set("spark.rapids.memory.pinnedPool.size", "4g")
)

# Try GPU first; if it crashes the JVM, fall back to CPU-only cleanly
try:
    spark = SparkSession.builder.config(conf=gpu).getOrCreate()
except Exception:
    cpu = (base
        .set("spark.rapids.sql.enabled", "false")
        .set("spark.plugins", ""))
    spark = SparkSession.builder.config(conf=cpu).getOrCreate()

print("Spark OK:", spark.version, "| rapids.enabled:", spark.conf.get("spark.rapids.sql.enabled", "<unset>"))


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/04 13:28:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/11/04 13:28:07 WARN RapidsPluginUtils: RAPIDS Accelerator 25.08.0 using cudf 25.08.0, private revision f4b467339f0ea78b7e2a862be97a63bc239e0b07
25/11/04 13:28:07 WARN RapidsPluginUtils: Multiple spark-rapids-jni jars found in the classpath:
revison: 155ef36a7e5c38e404d76976a61d177354c99281
	jar URL: jar:file:/users/chenqh23/spark/dist/jars/spark-rapids-jni-25.08.0.jar
	version=25.08.0
	user=root
	revision=155ef36a7e5c38e404d76976a61d177354c99281
	branch=HEAD
	date=2025-08-07T05:18:36Z
	url=https://github.com/NVIDIA/spark-rapids-jni.git
	gpu_architectures=100;120;70;75;80;86;90
	jar URL: jar:file:/users/chenqh23/spark/dist/jars/rapids-4-spark_2.12-25.08.0.jar
	version=25.08.0
	user=root
	revision=155ef36a7e5c38e404d

Spark OK: 3.5.6 | rapids.enabled: true


In [4]:
def runMicroBenchmark(spark, appName, query, retryTimes):
    count = 0
    total_time = 0
    # You can print the physical plan of each query
    # spark.sql(query).explain()
    while count < retryTimes:
        start = time.time()
        spark.sql(query).collect()
        end = time.time()
        total_time += round(end - start, 2)
        count = count + 1
        print("Retry times : {}, ".format(count) + appName + " microbenchmark takes {} seconds".format(round(end - start, 2)))
    print(appName + " microbenchmark takes average {} seconds after {} retries".format(round(total_time/retryTimes),retryTimes))
    with open('result.txt', 'a') as file:
        file.write("{},{},{}\n".format(appName, round(total_time/retryTimes), retryTimes))

In [5]:
dataRoot = "/users/chenqh23/spark-rapids-examples/datasets"

spark.read.parquet(dataRoot + "/tpcds/customer").createOrReplaceTempView("customer")
spark.read.parquet(dataRoot + "/tpcds/store_sales").createOrReplaceTempView("store_sales")
spark.read.parquet(dataRoot + "/tpcds/catalog_sales").createOrReplaceTempView("catalog_sales")
spark.read.parquet(dataRoot + "/tpcds/web_sales").createOrReplaceTempView("web_sales")
spark.read.parquet(dataRoot + "/tpcds/item").createOrReplaceTempView("item")
spark.read.parquet(dataRoot + "/tpcds/date_dim").createOrReplaceTempView("date_dim")

# Cache hot tables to reduce I/O contention during parallel runs
for t in ("customer","store_sales","catalog_sales","web_sales","item","date_dim"):
    try:
        spark.catalog.cacheTable(t)
    except Exception:
        pass
# Materialize caches once to warm up
for t in ("customer","store_sales","catalog_sales","web_sales","item","date_dim"):
    _ = spark.table(t).count()

print("-"*50)
time.sleep(2)

25/11/04 13:28:24 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
25/11/04 13:28:28 WARN MultiFileReaderThreadPool: Configuring the file reader thread pool with a max of 128 threads instead of spark.rapids.sql.multiThreadedRead.numThreads = 16


--------------------------------------------------


### Expand&HashAggregate
This is a microbenchmark about Expand&HashAggregate expressions running on the GPU. The query calculates the distinct value of some dimension columns and average birth year by different c_salutation of customers after grouping by c_current_hdemo_sk. You will see about 10x speedups in this query. Because an additional shuffle involved by the repartition operator in CPU mode. And GPUExpand and GPUHashAggregate is much faster than Expand and HashAggregate because GPU algorithms allow us to parallelize the computation and we can utilize most of the GPU cores. The tasks' duration in the third stage is less than one second but will cost 20x-40x while running on CPU. There will be a more significant performance improvement along with the increasing number of count distinct columns and aggregate functions.

In [6]:
spark.conf.set("spark.rapids.sql.explain", "NONE")

query0 = '''
select c_current_hdemo_sk,
count(DISTINCT if(c_salutation=="Ms.",c_salutation,null)) as c1,
count(DISTINCT if(c_salutation=="Mr.",c_salutation,null)) as c12,
count(DISTINCT if(c_salutation=="Dr.",c_salutation,null)) as c13,

count(DISTINCT if(c_salutation=="Ms.",c_first_name,null)) as c2,
count(DISTINCT if(c_salutation=="Mr.",c_first_name,null)) as c22,
count(DISTINCT if(c_salutation=="Dr.",c_first_name,null)) as c23,

count(DISTINCT if(c_salutation=="Ms.",c_last_name,null)) as c3,
count(DISTINCT if(c_salutation=="Mr.",c_last_name,null)) as c32,
count(DISTINCT if(c_salutation=="Dr.",c_last_name,null)) as c33,

count(DISTINCT if(c_salutation=="Ms.",c_birth_country,null)) as c4,
count(DISTINCT if(c_salutation=="Mr.",c_birth_country,null)) as c42,
count(DISTINCT if(c_salutation=="Dr.",c_birth_country,null)) as c43,

count(DISTINCT if(c_salutation=="Ms.",c_email_address,null)) as c5,
count(DISTINCT if(c_salutation=="Mr.",c_email_address,null)) as c52,
count(DISTINCT if(c_salutation=="Dr.",c_email_address,null)) as c53,

count(DISTINCT if(c_salutation=="Ms.",c_login,null)) as c6,
count(DISTINCT if(c_salutation=="Mr.",c_login,null)) as c62,
count(DISTINCT if(c_salutation=="Dr.",c_login,null)) as c63,

count(DISTINCT if(c_salutation=="Ms.",c_preferred_cust_flag,null)) as c7,
count(DISTINCT if(c_salutation=="Mr.",c_preferred_cust_flag,null)) as c72,
count(DISTINCT if(c_salutation=="Dr.",c_preferred_cust_flag,null)) as c73,

count(DISTINCT if(c_salutation=="Ms.",c_birth_month,null)) as c8,
count(DISTINCT if(c_salutation=="Mr.",c_birth_month,null)) as c82,
count(DISTINCT if(c_salutation=="Dr.",c_birth_month,null)) as c83,

avg(if(c_salutation=="Ms.",c_birth_year,null)) as avg1,
avg(if(c_salutation=="Mr.",c_birth_year,null)) as avg2,
avg(if(c_salutation=="Dr.",c_birth_year,null)) as avg3,
avg(if(c_salutation=="Miss.",c_birth_year,null)) as avg4,
avg(if(c_salutation=="Mrs.",c_birth_year,null)) as avg5,
avg(if(c_salutation=="Sir.",c_birth_year,null)) as avg6,
avg(if(c_salutation=="Professor.",c_birth_year,null)) as avg7,
avg(if(c_salutation=="Teacher.",c_birth_year,null)) as avg8,
avg(if(c_salutation=="Agent.",c_birth_year,null)) as avg9,
avg(if(c_salutation=="Director.",c_birth_year,null)) as avg10
from customer group by c_current_hdemo_sk
'''
print("-"*50)

--------------------------------------------------


In [7]:
# Run microbenchmark with n retry time
runMicroBenchmark(spark,"Expand&HashAggregate",query0,2)
time.sleep(2)

Retry times : 1, Expand&HashAggregate microbenchmark takes 4.86 seconds


Retry times : 2, Expand&HashAggregate microbenchmark takes 2.9 seconds
Expand&HashAggregate microbenchmark takes average 4 seconds after 2 retries


### Windowing(without data skew)
This is a microbenchmark about windowing expressions running on GPU mode. The sub-query calculates the average ss_sales_price of a fixed window function partition by ss_customer_sk, and the parent query calculates the average price of the sub-query grouping by each customer. You will see about 25x speedups in this query. The speedup mainly comes from GPUSort/GPUWindow/GPUHashAggregate. The avg aggregation function evaluates all rows which are generated by the sub-query's window function. There will be a more significant performance improvement along with the increasing number of sub-query aggregate functions.

In [8]:
query1 = '''
select ss_customer_sk,avg(avg_price) as avg_price
from
(
SELECT ss_customer_sk ,avg(ss_sales_price) OVER (PARTITION BY ss_customer_sk order by ss_sold_date_sk ROWS BETWEEN 50 PRECEDING AND 50 FOLLOWING ) as avg_price
FROM store_sales
where ss_customer_sk is not null
) group by ss_customer_sk order by 2 desc 
'''
print("-"*50)

--------------------------------------------------


In [9]:
# Run microbenchmark with n retry time
runMicroBenchmark(spark,"Windowing without skew",query1,2)
time.sleep(2)


Retry times : 1, Windowing without skew microbenchmark takes 9.49 seconds


Retry times : 2, Windowing without skew microbenchmark takes 8.22 seconds
Windowing without skew microbenchmark takes average 9 seconds after 2 retries


### Windowing(with data skew)
Data skew is caused by many null values in the ss_customer_sk column. You will see about 80x speedups in this query. The heavier skew task a query has, the more improved performance we will get because GPU parallelizes the computation, CPU is limited to just a single core because of how the algorithms are written.


In [10]:
query2 = '''
select ss_customer_sk,avg(avg_price) as avg_price
from
(
SELECT ss_customer_sk ,avg(ss_sales_price) OVER (PARTITION BY ss_customer_sk order by ss_sold_date_sk ROWS BETWEEN 50 PRECEDING AND 50 FOLLOWING ) as avg_price
FROM store_sales
) group by ss_customer_sk order by 2 desc 
'''
print("-"*50)


--------------------------------------------------


In [11]:
# Run microbenchmark with n retry time
runMicroBenchmark(spark,"Windowing with skew",query2,2)
time.sleep(2)


Retry times : 1, Windowing with skew microbenchmark takes 7.6 seconds


Retry times : 2, Windowing with skew microbenchmark takes 7.3 seconds
Windowing with skew microbenchmark takes average 7 seconds after 2 retries


### Intersection
This is a microbenchmark about intersection operation running on GPU mode. The query calculates items in the same brand, class, and category that are sold in all three sales channels in two consecutive years. You will see about 10x speedups in this query. This is a competition between high cardinality SortMergeJoin vs GpuShuffleHashJoin. The mainly improved performance comes from two SortMergeJoin(s) in this query running on CPU get converted to GpuShuffleHashJoin running on GPU.


In [12]:
query3 = '''
select i_item_sk ss_item_sk
 from item,
    (select iss.i_brand_id brand_id, iss.i_class_id class_id, iss.i_category_id category_id
     from store_sales, item iss, date_dim d1
     where ss_item_sk = iss.i_item_sk
                    and ss_sold_date_sk = d1.d_date_sk
       and d1.d_year between 1999 AND 1999 + 2
   intersect
     select ics.i_brand_id, ics.i_class_id, ics.i_category_id
     from catalog_sales, item ics, date_dim d2
     where cs_item_sk = ics.i_item_sk
       and cs_sold_date_sk = d2.d_date_sk
       and d2.d_year between 1999 AND 1999 + 2
   intersect
     select iws.i_brand_id, iws.i_class_id, iws.i_category_id
     from web_sales, item iws, date_dim d3
     where ws_item_sk = iws.i_item_sk
       and ws_sold_date_sk = d3.d_date_sk
       and d3.d_year between 1999 AND 1999 + 2) x
 where i_brand_id = brand_id
   and i_class_id = class_id
   and i_category_id = category_id
'''


In [13]:
# Run microbenchmark with n retry time
runMicroBenchmark(spark,"NDS Q14a subquery",query3,2)
time.sleep(2)


Retry times : 1, NDS Q14a subquery microbenchmark takes 6.63 seconds


Retry times : 2, NDS Q14a subquery microbenchmark takes 6.13 seconds
NDS Q14a subquery microbenchmark takes average 6 seconds after 2 retries


In [ ]:
# Run 8 micro-benchmarks concurrently on one SparkSession/one GPU
# Requires: query0, query1, query2, query3 already defined; temp views already created.

from concurrent.futures import ThreadPoolExecutor, as_completed
import time, os

# Increase GPU concurrency and reduce per-task batch size to fit 8 tasks
spark.conf.set("spark.rapids.sql.concurrentGpuTasks", os.environ.get("MB_CONCURRENT_GPU_TASKS", "8"))
spark.conf.set("spark.rapids.sql.batchSizeBytes", os.environ.get("MB_GPU_BATCH_BYTES", "128m"))
spark.conf.set("spark.sql.files.maxPartitionBytes", os.environ.get("MB_MAX_PART_BYTES", "128m"))

# Materialize DataFrames once to avoid SQL parsing overhead per retry
_dfA1 = spark.sql(query0)
_dfB1 = spark.sql(query1)
_dfC1 = spark.sql(query2)
_dfD1 = spark.sql(query3)
# Replicate each workload to double total jobs
_dfA2 = _dfA1
_dfB2 = _dfB1
_dfC2 = _dfC1
_dfD2 = _dfD1


def run_df_in_pool(pool_name: str, df, retryTimes: int = 2):
    sc = spark.sparkContext
    t0 = time.time()
    for i in range(retryTimes):
        sc.setLocalProperty("spark.scheduler.pool", pool_name)  # assign this job to a pool
        sc.setJobGroup(f"{pool_name}-{i+1}", f"{pool_name} run {i+1}", True)
        try:
            df.collect()  # blocking action submits a job
        finally:
            sc.setLocalProperty("spark.scheduler.pool", None)   # clear after the job
            sc.setJobGroup(None, None)
    return pool_name, round(time.time() - t0, 2)

jobs = [
    ("poolA", _dfA1),
    ("poolB", _dfB1),
    ("poolC", _dfC1),
    ("poolD", _dfD1),
    ("poolA2", _dfA2),
    ("poolB2", _dfB2),
    ("poolC2", _dfC2),
    ("poolD2", _dfD2),
]

# You can tune the parallelism here quickly if you hit contention
PARALLEL_JOBS = int(os.environ.get("MB_PARALLEL_JOBS", "8"))

with ThreadPoolExecutor(max_workers=PARALLEL_JOBS) as ex:
    futs = [ex.submit(run_df_in_pool, p, df) for p, df in jobs[:PARALLEL_JOBS]]
    for f in as_completed(futs):
        name, secs = f.result()
        print(f"{name} took {secs}s")


poolA took 20.79s


poolD took 25.12s
poolC took 30.17s
poolB took 30.49s
